# Synergy model interpretation for individual drug combinations

## Data loading

Configure root with local/colab.

In [ ]:
import sys, subprocess
import pandas as pd
%matplotlib inline
from pathlib import Path

# Configure root
COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/phenotype-prediction.git"
repo_dir = Path("phenotype-prediction")
if COLAB:
    root = Path("/content/phenotype-prediction")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])
else:
    root = Path.cwd().parent
sys.path.insert(0, str(root))

Configure data path for colab/local.

In [ ]:
if COLAB:
  from google.colab import drive
  drive.mount("/content/drive")
  data_dir = Path("/content/drive/MyDrive/phenotype-prediction-data")
  l2fc_dir = str(data_dir / "dge")
  cfu_dir = str(data_dir /  "cfus")

else:
  l2fc_dir = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/dge"
  cfu_dir = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/cfus"

Load the time-matched transcriptional scores, synergy scores, and metadata.

In [ ]:
from src.dge_data import (
    simple_interaction_score,
    eob_score,
    get_all_synergy_data
)

# Bliss score and simple interaction score
data_df = get_all_synergy_data(
    l2fc_dir = l2fc_dir,
    cfu_dir = cfu_dir,
    interaction_score_method = simple_interaction_score,
    synergy_score_method = eob_score,
    time_matched = True
)

# Drop genes with NA values
data_df = data_df.dropna(axis = 1)

## CEF+CIP feature interpretation

Feature importance strategies
- Find features consistently highkly ranked across CV folds
    - Rank using VIP scores
    - Find feature directionality using coefficients?
    - Try permutation importance?
    - Find features that were stable in the CV, then fit the model on the full data and see how many of those last?

"Features with stable high VIP scores across cross-validation and high importance in the final full-data model were prioritized as candidate mechanistic contributors to prediction."

Try permutation importance?

In [ ]:
def cv_feature_importances(
        df,
):
    """
    Running nested CV and gathering feature importances across folds
    """

    # Use random splitter


    # For each train, test idx, train a model on the training set

    # Extract feature importances from pipeline

    # Store as a column in dataframe


    # 

In [ ]:
# Reference code

def run_nested_pls_cv(
        df, 
        splits, 
        synergy
):
    """
    Running nested CV for PLS regression
    """

    scores = []

    if synergy:
        target = "synergy_score"
    else:
        target = "CFU"

    for train_idx, test_idx in splits:
        train_df = df.iloc[train_idx]
        X_train = train_df.iloc[:, train_df.columns.str.contains("SP")]
        y_train = train_df[target]

        test_df = df.iloc[test_idx]
        X_test = test_df.iloc[:, test_df.columns.str.contains("SP")]
        y_test = test_df[target]

        # Create a nested hyperparameter tuning scheme
        param_grid = {
            "model__n_components": list(range(3, 20))
        }
        
        # Make pipeline for PLS regression
        pipeline = Pipeline([
                ("scaler", StandardScaler()),
                ("model", PLSRegression())
        ])

        # Setup GridSearch
        search = GridSearchCV(
            estimator = pipeline,
            cv = 5,
            param_grid = param_grid,
            scoring = "neg_mean_squared_error",
        )

        # Fit with best params
        search.fit(X_train, y_train)
        preds = search.predict(X_test)

        # Evaluate
        score = r2_score(y_test, preds)
        scores.append(score)
    
    mean_score = np.mean(scores)

    # Round
    scores = [round(score, 3) for score in scores]
    mean_score = round(mean_score, 3)

    return scores, mean_score